# Seq2Seq Chatbot Training Notebook

Training loop for the clean-from-scratch Seq2Seq chatbot. Trains both **baseline** (no attention) and **attention** (Bahdanau) models with identical hyperparameters for a controlled ablation.

## Training Strategy

**Teacher Forcing (3-phase schedule):**
| Phase | Epochs | TF Ratio | Purpose |
|-------|--------|----------|---------|
| 1 — Foundation | 1-5 | 1.0 | Burn in basic token representations. Loss drops ~5.5 to ~4.25. |
| 2 — Annealing | 6-12 | 0.9 to 0.5 | Linear decay; decoder begins self-feeding while representations are refined. |
| 3 — Maturation | 13-20 | 0.5 | Hold at floor; both models fully adapt to semi-autoregressive generation. |

TF floor = 0.5 for both models (baseline decoder has no attention to recover from compounding errors; TF < 0.5 causes collapse).

**LR Strategy:** Cosine annealing with 500-step linear warmup (LR: 0 to 3e-4), then cosine decay to 1e-5. `scheduler.step()` is called per optimizer step, not per epoch.

**Early Stopping:** Patience = 4 epochs, monitored only from Phase 2 onward (epoch > 5).

**Gradient Clipping:** clip = 1.0. **AMP:** bf16 (no GradScaler needed).

In [ ]:
import os
import json
import math
import time
import sys
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

from config import CONFIG, get_tf_ratio, set_seed
from dataset import build_dataloaders
from models import build_model

## `train_epoch()` — One Training Epoch

Runs one training epoch with bf16 AMP and gradient accumulation. Returns `(avg_train_loss, avg_grad_norm, updated_global_step)`.

Key details:
- `global_step` counts optimizer steps, not forward passes
- bf16 does not underflow like fp16 -- GradScaler is not needed
- NaN loss: batch is skipped; NaN grad norm: optimizer step is skipped
- Periodic checkpoint every 2000 optimizer steps (atomic write)

In [ ]:
def train_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    config: dict,
    device: torch.device,
    epoch: int,
    writer: SummaryWriter,
    global_step: int,
    scheduler,
) -> Tuple[float, float, int]:
    """
    One training epoch with bf16 AMP and gradient accumulation.

    Returns:
        (avg_train_loss, avg_grad_norm, updated_global_step)
    """
    model.train()

    vocab_size: int = config["vocab_size"]
    grad_accum_steps: int = config["grad_accum_steps"]
    max_grad_norm: float = config["max_grad_norm"]
    checkpoint_dir: str = config["checkpoint_dir"]
    periodic_ckpt_steps: int = 2000
    _amp_dtype = getattr(torch, config.get("amp_dtype", "bfloat16"))

    tf_ratio: float = get_tf_ratio(epoch, config)

    total_loss = 0.0
    total_grad_norm = 0.0
    n_updates = 0
    nan_count = 0
    num_batches = len(loader)

    pbar = tqdm(
        enumerate(loader),
        total=num_batches,
        desc=f"  Epoch {epoch:3d} train",
        dynamic_ncols=True,
        unit="batch",
    )

    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in pbar:
        src: torch.Tensor = batch["src"].to(device)
        src_lengths: torch.Tensor = batch["src_lengths"].to(device)
        trg: torch.Tensor = batch["trg"].to(device)

        _device_type = device.type if hasattr(device, "type") else str(device).split(":")[0]
        with torch.amp.autocast(device_type=_device_type, dtype=_amp_dtype,
                                enabled=_device_type == "cuda"):
            output = model(src, src_lengths, trg, teacher_forcing_ratio=tf_ratio)
            loss = criterion(
                output.reshape(-1, vocab_size),
                trg[:, 1:].reshape(-1),
            )
            scaled_loss = loss / grad_accum_steps

        if not torch.isfinite(loss):
            nan_count += 1
            optimizer.zero_grad(set_to_none=True)
            pbar.set_postfix(loss="NaN", nan_skip=nan_count)
            continue

        scaled_loss.backward()

        is_last_batch = (batch_idx + 1) == num_batches
        should_step = ((batch_idx + 1) % grad_accum_steps == 0) or is_last_batch

        if should_step:
            grad_norm: float = torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_grad_norm
            ).item()

            if not math.isfinite(grad_norm):
                nan_count += 1
                optimizer.zero_grad(set_to_none=True)
                pbar.set_postfix(loss=f"{loss.item():.4f}", grad_norm="NaN", nan_skip=nan_count)
                continue

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            total_loss += loss.item()
            total_grad_norm += grad_norm
            n_updates += 1

            writer.add_scalar("train/loss_step", loss.item(), global_step)
            writer.add_scalar("train/grad_norm_step", grad_norm, global_step)
            writer.add_scalar("train/lr_step", optimizer.param_groups[0]["lr"], global_step)

            if global_step % periodic_ckpt_steps == 0:
                ckpt_path = os.path.join(
                    checkpoint_dir,
                    f"{config.get('_model_type', 'model')}_step_{global_step}.pt",
                )
                tmp_path = ckpt_path + ".tmp"
                torch.save(
                    {
                        "epoch": epoch,
                        "global_step": global_step,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "train_loss_so_far": total_loss / max(n_updates, 1),
                        "tf_ratio": tf_ratio,
                    },
                    tmp_path,
                )
                os.replace(tmp_path, ckpt_path)

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                grad_norm=f"{grad_norm:.3f}",
                nan_skip=nan_count,
            )

    avg_train_loss = total_loss / max(n_updates, 1)
    avg_grad_norm = total_grad_norm / max(n_updates, 1)
    return avg_train_loss, avg_grad_norm, global_step

## `evaluate_epoch()` — Validation Pass

Teacher forcing is **disabled** (ratio=0.0). `trg` is passed only to set the decoder step count. Loss is accumulated only on non-padding positions (`ignore_index=pad_idx`).

Returns `(avg_val_loss, val_ppl)` where `val_ppl = exp(min(avg_val_loss, 20))`.

In [ ]:
@torch.inference_mode()
def evaluate_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
    amp_dtype: torch.dtype = torch.bfloat16,
) -> Tuple[float, float]:
    """
    Validation pass — teacher forcing is DISABLED (ratio=0.0).
    Returns:
        (avg_val_loss, val_ppl) where val_ppl = exp(min(avg_val_loss, 20)).
    """
    model.eval()

    total_loss = 0.0
    n_batches = 0

    for batch in tqdm(loader, desc="  val", unit="batch", dynamic_ncols=True, leave=False):
        src: torch.Tensor = batch["src"].to(device)
        src_lengths: torch.Tensor = batch["src_lengths"].to(device)
        trg: torch.Tensor = batch["trg"].to(device)

        _device_type = device.type if hasattr(device, "type") else str(device).split(":")[0]
        with torch.amp.autocast(device_type=_device_type, dtype=amp_dtype,
                                enabled=_device_type == "cuda"):
            output = model(src, src_lengths, trg, teacher_forcing_ratio=0.0)

        vocab_size: int = output.size(-1)
        loss = criterion(
            output.reshape(-1, vocab_size),
            trg[:, 1:].reshape(-1),
        )

        if torch.isfinite(loss):
            total_loss += loss.item()
            n_batches += 1

    avg_val_loss = total_loss / max(n_batches, 1)
    val_ppl = math.exp(min(avg_val_loss, 20))
    return avg_val_loss, val_ppl

## `build_optimizer_and_scheduler()` — AdamW + Cosine Annealing

Builds the AdamW optimizer and a two-phase LR scheduler (chained via `SequentialLR`):
1. **Warmup** -- LR ramps linearly from ~0 to peak over `lr_warmup_steps` optimizer steps
2. **Cosine** -- LR decays from peak to `lr_min` over the remaining steps

`scheduler.step()` must be called once per optimizer step (inside `train_epoch`), NOT once per epoch.

In [ ]:
def build_optimizer_and_scheduler(
    model: nn.Module,
    config: dict,
    total_steps: int,
) -> Tuple[torch.optim.Optimizer, torch.optim.lr_scheduler.SequentialLR]:
    """
    Build AdamW optimizer and a cosine-annealing scheduler with linear warmup.
    Returns:
        (optimizer, scheduler)
    """
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    warmup_steps: int = config.get("lr_warmup_steps", 500)
    lr_min: float = config.get("lr_min", 1e-5)
    cosine_steps: int = max(total_steps - warmup_steps, 1)

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0 / warmup_steps,
        end_factor=1.0,
        total_iters=warmup_steps,
    )

    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cosine_steps,
        eta_min=lr_min,
    )

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps],
    )
    return optimizer, scheduler

## `train_model()` — Full Training Run

Runs the complete training loop for one model type (`"baseline"` or `"attention"`). Handles:
- Data loading, model construction, optimizer/scheduler setup
- Checkpoint resume (prefers last-epoch checkpoint for Colab resilience)
- Per-epoch train/validate cycle with TensorBoard logging
- Best checkpoint saving (atomic write), early stopping from Phase 2 onward
- History JSON export

In [ ]:
def train_model(model_type: str, config: dict, device: torch.device) -> Dict[str, List]:
    """
    Full training run for one model_type ("baseline" or "attention").
    Returns:
        Dict of per-epoch history lists.
    """
    config = dict(config)
    config["_model_type"] = model_type

    # 1. Data
    train_loader, val_loader, _ = build_dataloaders(
        artifact_dir=config["artifact_dir"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        max_ctx_len=config["max_ctx_tokens"],
        max_resp_len=config["max_resp_tokens"] + 2,
        pad_idx=config["pad_idx"],
        max_train_samples=config.get("max_train_samples", 0),
    )

    # 2. Model
    model = build_model(model_type, config, device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[{model_type}] Trainable parameters: {num_params:,}")

    # 3. Optimizer + scheduler
    total_steps = config["num_epochs"] * (len(train_loader) // config["grad_accum_steps"])
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, total_steps)

    # 4. Loss
    criterion = nn.CrossEntropyLoss(
        ignore_index=config["pad_idx"],
        label_smoothing=config.get("label_smoothing", 0.0),
    )
    _amp_dtype = getattr(torch, config.get("amp_dtype", "bfloat16"))

    # 5. TensorBoard
    tb_dir = os.path.join(config["tensorboard_dir"], model_type)
    writer = SummaryWriter(log_dir=tb_dir)

    # 6. Resume
    checkpoint_dir = config["checkpoint_dir"]
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_ckpt_path = os.path.join(checkpoint_dir, f"{model_type}_best.pt")
    last_ckpt_path = os.path.join(checkpoint_dir, f"{model_type}_last.pt")

    best_val_loss = float("inf")
    start_epoch = 1
    global_step = 0
    _patience: int = config.get("patience", 0)
    _no_improve: int = 0
    history: Dict[str, List] = {
        "train_loss": [],
        "val_loss": [],
        "tf_ratios": [],
        "lrs": [],
    }

    resume_path = last_ckpt_path if os.path.exists(last_ckpt_path) else (
        best_ckpt_path if os.path.exists(best_ckpt_path) else None
    )
    if resume_path:
        print(f"[{model_type}] Resuming from {resume_path}")
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        best_val_loss = ckpt.get("val_loss", float("inf"))
        start_epoch = ckpt.get("epoch", 0) + 1
        global_step = ckpt.get("global_step", 0)
        if "history" in ckpt:
            history = ckpt["history"]
        print(f"[{model_type}] Resumed at epoch {start_epoch}, step {global_step}, "
              f"best_val={best_val_loss:.4f}")

    num_epochs: int = config["num_epochs"]
    print(f"[{model_type}] Training epochs {start_epoch}–{num_epochs}")

    # 7. Training loop
    for epoch in range(start_epoch, num_epochs + 1):
        epoch_start = time.time()
        tf_ratio = get_tf_ratio(epoch, config)

        # Train
        train_loss, avg_gnorm, global_step = train_epoch(
            model=model, loader=train_loader, optimizer=optimizer,
            criterion=criterion, config=config, device=device,
            epoch=epoch, writer=writer, global_step=global_step,
            scheduler=scheduler,
        )

        # Validate
        val_loss, val_ppl = evaluate_epoch(
            model=model, loader=val_loader, criterion=criterion,
            device=device, amp_dtype=_amp_dtype,
        )

        elapsed = time.time() - epoch_start
        lr: float = optimizer.param_groups[0]["lr"]

        # TensorBoard epoch-level logging
        writer.add_scalar("val/loss", val_loss, epoch)
        writer.add_scalar("val/ppl", val_ppl, epoch)
        writer.add_scalar("train/loss_epoch", train_loss, epoch)
        writer.add_scalar("learning_rate", lr, epoch)
        writer.add_scalar("tf_ratio", tf_ratio, epoch)

        # Save best checkpoint (atomic)
        _improved = val_loss < best_val_loss
        if _improved:
            best_val_loss = val_loss
            try:
                _git = subprocess.check_output(
                    ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
                ).decode().strip()
            except Exception:
                _git = "unknown"
            ckpt_data = {
                "epoch": epoch, "global_step": global_step,
                "model_type": model_type,
                "model_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "val_loss": val_loss, "val_ppl": val_ppl,
                "train_loss": train_loss, "tf_ratio": tf_ratio,
                "config": dict(config), "history": history,
                "git_hash": _git, "torch_version": torch.__version__,
                "run_timestamp": time.strftime("%Y%m%d_%H%M%S"),
            }
            tmp_path = best_ckpt_path + ".tmp"
            torch.save(ckpt_data, tmp_path)
            os.replace(tmp_path, best_ckpt_path)

        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["tf_ratios"].append(tf_ratio)
        history["lrs"].append(lr)

        # Early stopping (only from Phase 2 onward)
        if _patience > 0 and epoch > config["tf_schedule"]["phase1_end"]:
            if _improved:
                _no_improve = 0
            else:
                _no_improve += 1
                if _no_improve >= _patience:
                    print(f"[{model_type}] Early stopping at epoch {epoch} "
                          f"(no improvement for {_patience} epochs)")
                    last_ckpt_data = {
                        "epoch": epoch, "global_step": global_step,
                        "model_type": model_type,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "val_loss": val_loss, "config": dict(config),
                        "history": history,
                    }
                    tmp_last = last_ckpt_path + ".tmp"
                    torch.save(last_ckpt_data, tmp_last)
                    os.replace(tmp_last, last_ckpt_path)
                    break

        # Save last-epoch checkpoint (for Colab resume)
        last_ckpt_data = {
            "epoch": epoch, "global_step": global_step,
            "model_type": model_type,
            "model_state_dict": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "val_loss": val_loss, "config": dict(config),
            "history": history,
        }
        tmp_last = last_ckpt_path + ".tmp"
        torch.save(last_ckpt_data, tmp_last)
        os.replace(tmp_last, last_ckpt_path)

        # Epoch summary
        print(
            f"Epoch {epoch:3d}/{num_epochs} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"PPL: {val_ppl:.2f} | "
            f"LR: {lr:.2e} | "
            f"TF: {tf_ratio:.2f} | "
            f"Grad: {avg_gnorm:.3f} | "
            f"{elapsed:.1f}s"
        )

    writer.close()

    # 8. Save history JSON (atomic write)
    history_path = os.path.join(checkpoint_dir, f"{model_type}_history.json")
    _tmp_hist = history_path + ".tmp"
    with open(_tmp_hist, "w") as fh:
        json.dump(history, fh, indent=2)
    os.replace(_tmp_hist, history_path)
    print(f"[{model_type}] History saved to {history_path}")

    return history

## Configuration

Device setup, seed, output directories, and run metadata. The `CONFIG` dict from `config.py` provides all hyperparameters. Edit values here to override defaults without modifying the source file.

In [ ]:
# Use CONFIG from config.py as the base; override individual keys as needed.
active_cfg = dict(CONFIG)

# ── Reproducibility ──────────────────────────────────────────────────────────
set_seed(active_cfg.get("seed", 42))

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Output directories ───────────────────────────────────────────────────────
os.makedirs(active_cfg["checkpoint_dir"], exist_ok=True)
os.makedirs(active_cfg["tensorboard_dir"], exist_ok=True)

# ── Run metadata ─────────────────────────────────────────────────────────────
try:
    git_hash = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    git_hash = "unknown"

run_info = {
    "git_hash":       git_hash,
    "python_version": sys.version,
    "torch_version":  torch.__version__,
    "cuda_version":   torch.version.cuda or "cpu",
    "run_timestamp":  time.strftime("%Y%m%d_%H%M%S"),
    "seed":           active_cfg.get("seed", 42),
    "device":         str(device),
}
run_info_path = os.path.join(active_cfg["checkpoint_dir"], "run_info.json")
with open(run_info_path, "w") as fh:
    json.dump(run_info, fh, indent=2)

print(f"Run info saved -> {run_info_path}")
print(f"  git: {git_hash}  |  torch: {torch.__version__}  |  seed: {run_info['seed']}")

## Train Baseline Model (No Attention)

Trains the baseline Seq2Seq model without attention. Uses the same hyperparameters and TF schedule as the attention model for a controlled comparison.

In [ ]:
print("=" * 70)
print("  TRAINING: baseline (no attention)")
print("=" * 70)

baseline_history = train_model("baseline", active_cfg, device)

## Train Attention Model (Bahdanau)

Trains the Seq2Seq model with Bahdanau attention. Identical hyperparameters and TF schedule as the baseline -- the only difference is the attention mechanism.

In [ ]:
print("=" * 70)
print("  TRAINING: attention (Bahdanau)")
print("=" * 70)

attention_history = train_model("attention", active_cfg, device)

## Summary

Compare best validation loss for both models and display the training history.

In [ ]:
print("\nTraining complete.")
print(f"  Baseline  best val loss : {min(baseline_history['val_loss']):.4f}")
print(f"  Attention best val loss : {min(attention_history['val_loss']):.4f}")

print("\n--- Baseline History ---")
for i, (tl, vl, tf, lr) in enumerate(zip(
    baseline_history["train_loss"], baseline_history["val_loss"],
    baseline_history["tf_ratios"], baseline_history["lrs"]
), 1):
    print(f"  Epoch {i:2d}  train={tl:.4f}  val={vl:.4f}  tf={tf:.2f}  lr={lr:.2e}")

print("\n--- Attention History ---")
for i, (tl, vl, tf, lr) in enumerate(zip(
    attention_history["train_loss"], attention_history["val_loss"],
    attention_history["tf_ratios"], attention_history["lrs"]
), 1):
    print(f"  Epoch {i:2d}  train={tl:.4f}  val={vl:.4f}  tf={tf:.2f}  lr={lr:.2e}")